In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from simulate.module import Module
from simulations import cycle, dcir

In [ ]:
module = Module(
    n_parallel=5,
    n_series=3,
    cell_parameters={
        "Std SOH / %": 2,
        "Std SOR / %": 11,
        "Nominal capacity / Ah": 5.0,
        "Initial SOC / %": 20,
    },
    bussbar_parameters={
        "Positive terminal relative width / 1": 10,
        "Negative terminal relative width / 1": 10,
        "Mean series resistance / Ohm": 1e-4,
        "Std series resistance / Ohm": 1e-5,
        "Mean parallel resistance / Ohm": 1e-9,
        "Std parallel resistance / Ohm": 1e-9,
    },
    interpolants={
        "Open-circuit voltage / V": "lut",
        "R0 / Ohm": "spline",
        "R1 / Ohm": "spline",
        "R2 / Ohm": "spline",
        "Tau1 / s": "spline",
        "Tau2 / s": "spline",
    },
    interpolants_options={"Tau1 / s": {"k": 2, "n": 10}},
)

TypeError: Interpolant.__init__() got an unexpected keyword argument 'k'

In [ ]:
# Plot the interpolants
# Dear god, why did I use plotly for this?
# This graph is something matplotlib handles isis pisi,
# but nooooo, plotly requires manual setting of spacing to avoid axis-label overlap with adjacent plots.
keys = list(module.cellgrid.interpolators.keys())
fig = make_subplots(
    rows=1,
    cols=len(keys),
    shared_xaxes=True,
    horizontal_spacing=0.5 / len(keys),
)
soc = np.linspace(0, 1, 100)
colors = {"Charge": "blue", "Discharge": "red"}
palette = px.colors.qualitative.Plotly
for i, key in enumerate(keys):
    for mode, interp in module.cellgrid.interpolators[key].items():
        fig.add_trace(
            go.Scatter(
                x=soc,
                y=interp(soc),
                name=mode,
                legendgroup=mode,
                showlegend=(i == 0),
                line=dict(color=colors[mode]),
            ),
            row=1,
            col=i + 1,
        )
    fig.update_xaxes(title_text="State of Charge / 1", row=1, col=i + 1)
    fig.update_yaxes(title_text=key, row=1, col=i + 1)
fig.show()


c:\code\github\Milestone-8\.venv\Lib\site-packages\scipy\interpolate\_interpolate.py:501: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]


In [ ]:
sim = dcir(module)

In [ ]:
fig = make_subplots(rows=7, cols=1, shared_xaxes=True)
for i in range(module.n_series):
    for j in range(module.n_parallel):
        fig.add_trace(
            go.Scatter(x=sim.t, y=sim.cell_current[:, i, j]),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(x=sim.t, y=sim.cell_voltage[:, i, j]),
            row=2,
            col=1,
        )
        fig.add_trace(
            go.Scatter(x=sim.t, y=sim.cell_soc[:, i, j]),
            row=3,
            col=1,
        )
        fig.add_trace(
            go.Scatter(x=sim.t, y=sim.cell_pol1[:, i, j]),
            row=4,
            col=1,
        )
        fig.add_trace(
            go.Scatter(x=sim.t, y=sim.cell_pol2[:, i, j]),
            row=5,
            col=1,
        )
        fig.add_trace(
            go.Scatter(x=sim.t, y=sim.cell_hyst[:, i, j]),
            row=6,
            col=1,
        )
fig.update_layout(showlegend=False, height=1500, width=800)
fig.show()

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True)
fig.add_trace(
    go.Scatter(x=sim.t, y=sim.module_current),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=sim.t, y=sim.module_voltage),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(x=sim.t, y=sim.module_soc),
    row=3,
    col=1,
)
fig.update_layout(showlegend=False)
fig.show()